In [3]:
# ============================================================
# SMART IRRIGATION SYSTEM - ADVANCED RAINFALL PREDICTION
# Deep Learning Style Pipeline + Custom Model
# ============================================================

import pandas as pd
import numpy as np
import pickle

# ============================================================
# ETL + DATA PREPROCESSING PIPELINE
# ============================================================

class RainfallETL:

    def __init__(self, file_path):
        self.file_path = file_path

    # ---------------- EXTRACT ----------------
    def extract(self):
        print("\n[STEP 1] DATA EXTRACTION")
        df = pd.read_csv(self.file_path)
        print("Shape:", df.shape)
        print(df.head(), "\n")
        return df

    # ---------------- CLEANING ----------------
    def clean(self, df):
        print("[STEP 2] DATA CLEANING")

        # Remove duplicates
        before = df.shape[0]
        df = df.drop_duplicates()
        after = df.shape[0]
        print(f"Duplicates removed: {before - after}")

        # Handle missing values
        missing = df.isnull().sum().sum()
        print("Missing values:", missing)

        # Fill numeric missing values with median
        for col in df.select_dtypes(include=np.number).columns:
            df[col].fillna(df[col].median(), inplace=True)

        # Fill categorical with mode
        for col in df.select_dtypes(include='object').columns:
            df[col].fillna(df[col].mode()[0], inplace=True)

        print("Cleaning completed\n")
        return df

    # ---------------- OUTLIER REMOVAL ----------------
    def remove_outliers(self, df):
        print("[STEP 3] OUTLIER REMOVAL (IQR METHOD)")

        numeric_cols = df.select_dtypes(include=np.number).columns

        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            df = df[(df[col] >= lower) & (df[col] <= upper)]

        print("Outliers removed. New shape:", df.shape, "\n")
        return df

    # ---------------- FEATURE ENGINEERING ----------------
    def feature_engineering(self, df):
        print("[STEP 4] FEATURE ENGINEERING")

        # Interaction features
        df["temp_humidity"] = df["temperature"] * df["humidity"]
        df["elev_wind"] = df["elevation"] * df["wind_speed"]

        # Log transform (to stabilize rainfall distribution)
        df["rainfall_log"] = np.log1p(df["rainfall"])

        print("New features created\n")
        return df

    # ---------------- ENCODING ----------------
    def encode(self, df):
        print("[STEP 5] ENCODING")

        if "soil_texture" in df.columns:
            df["soil_texture"] = df["soil_texture"].astype("category").cat.codes

        if "aspect" in df.columns:
            df["aspect"] = df["aspect"].astype("category").cat.codes

        print("Encoding completed\n")
        return df

    # ---------------- NORMALIZATION ----------------
    def normalize(self, X):
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0)

        return (X - self.mean) / (self.std + 1e-8)

    # ---------------- TRANSFORM ----------------
    def transform(self, df):
        print("[STEP 6] FINAL TRANSFORMATION")

        features = [
            "temperature",
            "humidity",
            "solar_radiation",
            "wind_speed",
            "elevation",
            "temp_humidity",
            "elev_wind"
        ]

        X = df[features].values
        y = df["rainfall_log"].values

        X = self.normalize(X)

        print("Transformation complete\n")
        return X, y


# ============================================================
# CUSTOM MODEL (ADVANCED GRADIENT DESCENT)
# ============================================================

class CustomRainfallModel:

    def __init__(self, lr=0.001, epochs=2000):
        self.lr = lr
        self.epochs = epochs

    def initialize(self, n_features):
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

    def loss(self, y, y_pred):
        return np.mean((y - y_pred) ** 2)

    def fit(self, X, y):
        print("[STEP 7] MODEL TRAINING\n")

        n_samples, n_features = X.shape
        self.initialize(n_features)

        for epoch in range(self.epochs):

            y_pred = self.predict(X)

            dw = (-2/n_samples) * np.dot(X.T, (y - y_pred))
            db = (-2/n_samples) * np.sum(y - y_pred)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if epoch % 250 == 0:
                print(f"Epoch {epoch} | Loss: {self.loss(y, y_pred):.5f}")

        print("\nTraining completed\n")

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self, f)




def train_pipeline():

    etl = RainfallETL("Crop recommendation dataset.csv")

    df = etl.extract()
    df = etl.clean(df)
    df = etl.remove_outliers(df)
    df = etl.feature_engineering(df)
    df = etl.encode(df)

    X, y = etl.transform(df)

    # Split
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = CustomRainfallModel(lr=0.001, epochs=2000)
    model.fit(X_train, y_train)

    # Evaluation
    preds = model.predict(X_test)
    mse = np.mean((y_test - preds) ** 2)

    print("[STEP 8] EVALUATION")
    print("MSE:", mse, "\n")

    model.save("rainfall_model.pkl")

    return model, etl


# ============================================================
# PREDICTION + IRRIGATION
# ============================================================

def predict_rainfall(model, etl, input_data):

    input_data = np.array(input_data)
    input_data = (input_data - etl.mean) / (etl.std + 1e-8)

    pred_log = model.predict(input_data.reshape(1, -1))

    # Convert back from log
    return np.expm1(pred_log[0])


def irrigation_decision(rainfall, soil_moisture):

    if rainfall < 5 and soil_moisture < 30:
        return "Irrigation Required"
    elif rainfall < 5:
        return "Soil Moisture OK"
    else:
        return "Rain Expected - No Irrigation"


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    model, etl = train_pipeline()

    sample = [30, 70, 500, 10, 300, 2100, 3000]

    rain = predict_rainfall(model, etl, sample)

    print(f"Predicted Rainfall: {rain:.2f} mm")

    decision = irrigation_decision(rain, soil_moisture=25)

    print("Decision:", decision)


[STEP 1] DATA EXTRACTION
Shape: (10000, 17)
    N   P   K  temperature   humidity         ph    rainfall    elevation  \
0  83   0  18    18.616243  53.506692   3.237846  259.231858  1747.653421   
1  62  22  93    10.331711  26.172273  10.253547  164.638752   163.765837   
2   9  63  55     5.055778  74.234192   9.967836  326.728287   472.977687   
3  59  41  26    42.680282  12.162952  13.208322  421.587338  3676.731479   
4  66   2  10    -6.553443  29.800571  12.641792  292.385076  3533.155816   

       slope aspect  water_holding_capacity  wind_speed  solar_radiation  \
0  11.186370  South               18.054481   21.171867       540.550329   
1  59.698539   East                1.375326   49.876971       401.616758   
2  88.301496   West               80.322260   10.713521       808.051278   
3  37.531847  South                0.737539   51.872135       743.594046   
4  87.941712   West               85.456789   95.960097       650.214772   

  soil_texture        ec         zn

C:\Users\My\AppData\Local\Temp\ipykernel_17316\2645142263.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\My\AppData\Local\Temp\ipykernel_17316\2645142263.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col].fillna(df[col].median(), inplace=True)
C:\Users\My\AppData\Local\Tem